# Thesis Running Log - Data Pipeline & Methodology

A consolidated record of the data pipeline decisions and their rationale, kept as
the working source for the methods chapter. Organised as: Joining → User
Segmentation → Churn Definition → Cleaning → Splitting & Imputation → Feature
Engineering → Modelling → Decisions Log.

> Items marked **[updated]** were revised to match the final code and supersede
> earlier notes.

---

## 1. Data Augmentation - Joining

The vehicle dataframe is merged with the activity dataframe via a join keyed on
**user_id and vehicle_id together**. Both keys are required because the same
physical car can produce an identical vehicle_id across users; keying on
vehicle_id alone would create duplicates.

### Join strategy considerations

**Inner join (rejected).** Discards both vehicle_ids with no activity and
activities with no vehicle identifier. The problem: removing rows with
unidentified vehicles disproportionately strips professional users (who scan many
cars, more of which go unidentified), pushing them toward looking "personal" and
corrupting the segmentation. *(Worth stating explicitly in the thesis.)*

**Right join (chosen).** Keeps all activity rows. This avoids dropping users who
have a vehicle_id in the activity data but are missing from the vehicle metadata,
which matters for segmentation - premature vehicle removal can force genuine
professional users to register as personal. It also discards vehicle-dataset rows
that carry a user_id and vehicle_id but no activity, under the assumption that no
activity for a vehicle means the car was not used within the observation window.

### Pre-join filtering

Users absent from the vehicle dataset are removed **before** the join: even when
such a user has vehicle_ids in the activity data, those vehicles' origin is
untraceable (no make/model/age metadata to carry downstream).

---

## 2. Defining Customer Groups (Personal vs Professional)

### Inverse HHI intuition

Inverse HHI (1 / HHI) reads as "uses X cars equally":

| Usage pattern | HHI | Effective vehicles |
|---|---|---|
| 100% one car | 1 | 1 |
| 50 / 50 two cars | 1/2 | 2 |
| 33 / 33 / 33 | 1/3 | 3 |
| 80 / 20 | 0.68 | 1.47 |
| 60 / 20 / 20 | 0.44 | 2.27 |

Bounds: 1/n ≤ HHI ≤ 1, equivalently n ≥ 1/HHI ≥ 1.

### Segmentation rule

A user is classified **personal** if **any** of three conditions holds (logical OR):

1. **Effective car rate** (1/HHI) ≤ threshold - set from the 80-90% quantile of
   effective car rates, since most users are personal.
2. **Top-4 car share** ≥ 80% of usage.
3. **Unique vehicles per user** ≤ the 95th percentile (≈ 20 cars; verify before
   applying).

### Why three OR'd conditions - each covers the others' blind spot **[updated]**

This was the key design point. The three are not redundant; each catches a failure
mode of the others:

- **Effective HHI** - accurate at low-to-mid unique-vehicle counts, but drifts
  toward its upper limit as unique-vehicle count grows, so it under-flags
  high-volume users.
- **Top-4 car share (80%)** - added specifically to combat that HHI drift: a
  high-volume user who still concentrates usage on a few cars is caught here.
- **95th-percentile unique-vehicle filter** - catches users with low vehicle
  counts but very evenly distributed usage, which both HHI and car share miss.

OR'ing them errs on the side of inclusion in the personal group. The decision rule
was confirmed against a figure. *(Justification framing: each criterion targets a
specific failure mode of the others - cleaner than a single arbitrary threshold.)*

### Burst removal before HHI / car-share **[updated]**

Bursty activity is collapsed before computing HHI and car share (grouped by
user_id, vehicle_id), because bursts inflate the apparent usage of specific
vehicles - producing artificially low effective HHI and optimistically high car
share. **Zero-gap rule:** rows with an identical timestamp (gap = 0) are *not*
treated as bursts - they are simultaneous distinct actions, not a rapid repeat
session - so only positive sub-threshold gaps collapse.

Right-censoring is assumed throughout for simplicity, even though a user in reality
churns somewhere between two observed time points.

---

## 3. Churn Definition

Churn is a **hard inactivity threshold of X days**, derived per client group from
the gap-between-sessions distribution.

- **X = 180 days** for personal users - wide enough to absorb seasonal behaviour
  shifts while still flagging genuine inactivity (70 days is the 95th percentile).
- Gaps are computed only over activities separated by **> BURST_TIME_HR** (a constant value set in the cleaning.py constants folder), so bursts of
  usage do not register as activity.
- Two churn cases are both labelled: **mid-sequence** (a gap between consecutive
  activities exceeds the threshold) and **end-of-sequence** (last activity is more
  than threshold days before the dataset max date).
- After the first churn trigger, all subsequent rows for that user are dropped -
  they would represent a post-churn state the model should not see.
- The **churn-adjusted date** is the activity date pushed forward by the threshold
  on triggering rows, so the interval grid covers the full at-risk window rather
  than stopping at the last observed activity.

Threshold selection also drew on seasonality: usage dips slightly in warmer
periods, a difference large enough to be worth reflecting in the threshold.

---

## 4. Cleaning - Assumptions & Step Order

### General assumptions

1. Users observed for too short a span (14 or 28 days) are excluded as unrepresentative
   (see step 5, **[updated]** below).
2. The user base is split into personal vs professional (HHI + top-n share + number of vehicles).
3. Churn threshold judged from the (95th × 2) percentile boundary per group and
   from where the usage distribution flattens.
4. removes users who joined too close to the end of the window 
    and have not had sufficient time to churn - otherwise they 
    inflate the estimated survival rate with no churn signal.
   $$S(T>=t) = \frac{\text{survivors + later joiners}}{\text{at risk + late joiners}}$$

   The above equation, simplified as it is, shows that later joiners inflate the survival rate. This is because the numerator and denominator increase by a constant of the number of later joiners and in the limit converges to 1 (since $S(t)<1$):
   $$ lim_{n \rightarrow \infin} \frac{\text{survivors} + n}{\text{at risk} + n} = lim_{n \rightarrow \infin} \frac{n(\frac{\text{survivors}}{n} + 1)}{n(\frac{\text{at risk}}{n} + 1)} = 1$$
5. Both mid- and end-of-sequence churn are marked.

### Step order (as executed in code) **[updated]**

0. Filter vehicle-dataset users who have multiple rows with **different** vehicles
   under an **identical** vehicle_id - can't attribute behaviour, treated as
   anomaly. (Removed because there was only a single user with such an anomaly)
1. Right join (on user_id + vehicle_id).
2. Filter users not present in the vehicle dataset.
3. Remove duplicate rows.
4. Fill missing vehicle_end_year with the current year; add a
   `still_in_production` boolean derived from the **original** missingness before
   filling.
5. Filter by user type (personal | professional).
6. Filter rows with missing vehicle metadata - done **after** user-type and
   end-year steps so the split isn't distorted by premature row loss.
7. (After splitting) impute vehicle_start_year - see §5.
8. Apply inactivity threshold filtering + the (max date − threshold) cutoff.
9. **Filter early churners [updated]** - previously "filter one-day users." Now
   removes users whose activity span is **< MIN_ACTIVITY_SPAN_DAYS (28)**, using
   `>=` so a user spanning exactly one full interval is **kept**. Rationale below.

NOTE: Filter early churners works only when churn has been applied to actualy users. Therefore, the filter must go AFTER inactivity thresholding

### Why "early churner" filtering, and why 28 days **[updated]**

A user who churns inside their first interval has only a **partial window** of
observed behaviour. Every windowed feature (drifts, rolling proportions,
session/action counts) assumes a complete interval behind it; built on a partial
window it is **measurement truncation, not signal**. Keeping such rows teaches the
model "near-empty feature row → churn," which is really "short observation →
churn" - a tautology that won't generalise.

Additionally, filtering vehicle metadata's NaN rows raises a risk of wrongly assuming that a user is an early churner. This can happen for rows that have missing vehicle metadata, but lie outisde of the early churn window. One can only acknoweldege the risk as there is no correct modelling approach and, thus, in this analysis, missing vehicle metadata rows are removed.

**Stated scope condition:** the model estimates churn risk *conditional on having
survived at least one complete interval*. The 28-day span threshold must stay
aligned with INTERVAL_IN_DAYS - both encode "one complete interval." If the
interval width changes, this threshold should track it.

---

## 5. Imputation & Splitting **[updated - major revision]**

### Revised approach: whole-dataset imputation in Python, split-and-model in R

The pipeline was originally designed with a leak-free train-only KNN imputation
performed after a stratified train/val/test split in Python. This has been
**replaced** with a simpler two-step arrangement:

1. **KNN imputation of `vehicle_start_year` on the entire dataset in Python**,
   before any split. The imputer, one-hot encoder (make) and ordinal encoder
   (mileage, low→high bucket order) are fit on all rows.
2. **Split and modelling are performed in R**, where the survival tooling
   (counting-process Cox via `survival::coxph`, penalized Cox via `glmnet`,
   tie-handling options, `pec` / `timeROC` for time-dependent metrics) is more
   complete than what is available in Python for time-varying-covariate survival.

### Consequences and honest disclosure

Imputing on the whole dataset introduces a mild information leak: the imputer
has seen validation/test rows when learning the neighbour structure used to fill
missing years. This leak is:

- Confined to a single covariate (`vehicle_start_year`), not the churn label.
- Feature-level, not outcome-level, so it cannot inflate the discrimination
  metric via label knowledge.
- Uniform across users, since imputation runs on unique vehicles and maps
  back - no per-row cross-fold contamination.

The alternative (re-imputing per CV fold in R) was considered and rejected on
practical grounds: reimplementing the KNN + encoder stack inside R's
resampling loop is a substantial effort for a small, feature-only bias. The
tradeoff is acknowledged explicitly in the methods chapter and treated as a
limitation rather than concealed. This is standard practice in applied survival
work when the imputed variable is a covariate rather than the outcome.

### Stratified sampling removed **[updated]**

The user-level split stratified on first-activity year (with rare-year binning)
has been removed. All splitting is now performed in R using its native
resampling tools (either `rsample::group_vfold_cv` for user-grouped CV or a
simple grouped split). Rationale for the change:

- The stratification handled rare-year classes that could break `train_test_split`
  in scikit-learn. R's resampling packages handle grouped resampling without
  requiring that pre-binning step.
- The modelling loop is entirely inside R.

### Imputation details (retained)

- **Mileage is ordinal**, encoded with an explicit low→high `categories` list
  derived from the data, sorted by each bucket's lower edge:
  `0-50000 < 50000-100000 < 100000-150000 < ... < 500000-1000000 < >1000000`.
  Unknowns encoded as -1 (`handle_unknown="use_encoded_value"`).
- **Make is nominal**, one-hot encoded with `handle_unknown="ignore"` so an
  unseen make encodes as all-zeros and the feature-matrix width stays stable.
- Imputation runs on unique vehicles then maps back to all activity rows -
  identical vehicles would otherwise be re-imputed redundantly.

---

## 6. Feature Engineering

Built in Polars for performance. Produces a per-interval counting-process
frame with time-varying covariates.

### Process

1. Generate per-user intervals between each user's min and max date
   (`date_ranges`).
2. `join_asof` (backward) anchors each activity to the most recent interval start
   at or before its date.
3. Left-join the full interval grid back so empty intervals (no activity) are
   reintroduced - an empty interval is itself the signal of interest.
4. Per-interval intermediate aggregates via group_by (user_id, interval_start).
5. Staged post-aggregation passes with configurable lookback windows (each stage
   references columns the previous stage created, so they are applied as
   successive `with_columns` passes).

### Feature notes

- Make **proportions** (`prop_vw`, `prop_bmw_group`, …) are kept instead of a
  primary-make-group flag - they carry the same information more richly and avoid
  multicollinearity (the two together are perfectly collinear).
- `n_unique_vehicles` is irrelevant for personal users - fleet sizes were already
  capped by segmentation.
- `mean_car_age` deliberately counts each distinct vehicle once per user-day
  (`is_first_distinct` over [user_id, activity_date], excluding the "unknown"
  fill), scoped on **activity_date** not churn_adjusted_date (usage is on the real
  date; churn date only shapes the interval grid).
- App features capture **proportional drift** between apps (external switching).
  Only the main app's windowed proportion is built - with two apps the other is
  its complement, so both would be collinear.
- `prop_main` over 56 days differs from the drift feature: the 56-day value is a
  level, the drift captures the shift from the last-28 vs the prior-28 window.
- **Duration → days:** recency (`interval_end − last_activity_date`) is cast with
  `.dt.total_days()` to an integer - Polars can't write a Duration to CSV, and
  days is the feature you actually want.
- **Underflow guard:** session/action counts are unsigned; drift differences are
  cast to `Int32` before subtraction so a falling drift doesn't underflow to a
  large positive number.

### `.over()` discipline **[new]**

Any position- or accumulation-dependent expression must be scoped per user **and**
the rows must be sorted within user first (the two are a pair):

- **Needs `.over(USER_ID_COL)`** (+ prior sort): `shift`, `cum_sum`, `cum_count`,
  `cummax`, `forward_fill`, `diff`, per-user min/max flags.
- **Does not need it** (row-wise): `a − b`, `a / b`, `fill_null`, `cast`,
  `fill_nan`, comparisons.
- Rule of thumb: *if removing a row above or below would change the result, you
  need `.over` and a sort.*

Placement: `.col(...).<positional_op>().over(user).<row-wise cleanup>().alias(...)`
- `.over` hugs the positional op; row-wise calls follow.

### Churn label construction **[new]**

- Per interval, the churn flag is `.max()` (interval is churned if churn fired
  anywhere inside it).
- The label is **shifted back one interval** (`shift(-1).over(user)`): row *t*
  carries "did the user churn in *t+1*", matching the counting-process convention
  (covariates from *t*, outcome at the boundary). Depends on the prior
  `[user, interval_start]` sort.
- Each user's **last interval** is dropped as incomplete - its real churn label
  was already shifted onto the prior row.


---

## 7. Churn Models **[updated - now performed in R]**

Modelling and split generation are performed in R for its more complete
survival-analysis ecosystem. The Python pipeline exports the imputed
person-interval frame as CSV; R reads it and handles splits, resampling,
model fitting, and evaluation.

### Discrete-time

Would require an extra binary covariate per interval; for finer intervals this
becomes impractical.

### Continuous-time (Cox PH) - leaning toward

Concern: naive Cox PH discards information. Addressed by the **counting-process /
time-varying-covariate** format - the interval frame carries last-value-forward at
each risk set, and history is feature-engineered into each interval (rolling means,
slopes, recency, windowed counts), which is sound and supported by the reference
paper.

### Why R for modelling

- `survival::coxph` natively takes `Surv(start, stop, event)` counting-process
  format with time-varying covariates - the format Python survival libraries
  handle awkwardly or not at all (scikit-survival requires one row per subject;
  lifelines has `CoxTimeVaryingFitter` but with limited regularization support).
- `ties = "efron" | "breslow" | "exact"` is exposed directly on `coxph`.
- `cluster(user_id)` produces sandwich/robust standard errors that account for
  within-user dependence across intervals.
- `pec`, `timeROC`, `riskRegression` provide time-dependent Brier / IBS /
  cumulative-dynamic AUC natively for counting-process input.
- `glmnet::coxnet` for regularized Cox (elastic net) with `cv.glmnet` for
  penalty tuning.

### Handling separation

Because churn is defined as 180 days of inactivity, feature windows overlapping
the pre-churn quiet period contain the event by construction, producing
(quasi-)complete separation of the affected covariates from the outcome. This is
being addressed at the **feature-engineering level** (restricting interval rows
to the user's active period rather than the churn-adjusted grid), not by
penalization alone - penalization would stabilize the estimates but the
underlying tautology would remain.

---

## 8. Decisions Log (one-line index)

- **Join** - right join on (user_id, vehicle_id); inner join biases professionals
  toward personal. Remove vehicle-absent users pre-join.
- **Segmentation** - personal if ANY of {1/HHI ≤ ~5, top-4 share ≥ 80%, ≤ 95th pct
  unique vehicles}; three OR'd because each covers the others' blind spot.
- **Burst collapse before HHI/share** - zero-gap rows kept (simultaneous, not
  bursts); only positive sub-threshold gaps collapse.
- **Churn threshold** - 180d personal, from gap distribution + seasonality;
  gaps over >1h activity only.
- **Churn date push-forward** - adjusted date = activity date + threshold on
  triggers, so the grid covers the at-risk window.
- **Early-churner filter** - drop span < 28d, `>=` keeps exactly-one-interval
  users; scope = "conditioned on ≥1 complete interval." Must track INTERVAL_IN_DAYS.
- **Metadata NaN filter after segmentation** - avoid distorting the split.
- **Split moved to R [updated]** - user-grouped resampling in R (no longer
  stratified on first year in Python); one-time split saved to CSV.
- **KNN imputation on whole dataset [updated]** - fit once in Python on all rows
  (unique vehicles then map back); ordinal mileage (explicit order, unknown=-1),
  one-hot make (handle_unknown="ignore"); mild feature-level leak accepted for
  simplicity vs re-imputing per fold in R.
- **group_by ordering** - sort by user_id before to_pandas() in feature
  engineering; group_by isn't order-stable. Persist splits to disk from R.
- **Recency → total_days** - Duration can't be written to CSV.
- **Drift Int32 cast** - prevents unsigned underflow on falling drift.
- **.over discipline** - positional ops scoped per user + sorted; row-wise ops not.
- **Churn label** - per-interval max, shift(-1) over user, drop incomplete last
  interval.
- **First-interval churner rescue** - when/then keeps label on first interval;
  last-interval drop excepts first-and-last churned rows. Confirmed load-bearing.
- **Modelling in R [updated]** - counting-process Cox via `coxph`, penalized via
  `glmnet::coxnet`, time-dependent metrics via `pec` / `timeROC`; chosen over
  Python for native counting-process support and mature survival tooling.
- **Separation addressed at feature level [updated]** - restrict interval rows
  to the user's active period so the 180-day quiet period isn't included in
  covariate windows; penalization alone would mask a structural tautology.

---

## 9. Open Items / TODO

- [ ] Tie MIN_ACTIVITY_SPAN_DAYS and INTERVAL_IN_DAYS to a single shared constant.
- [ ] Windowing sensitivity table: window length (14 vs 28) → churners retained
      (event retention vs feature stability tradeoff).
- [ ] Confirm professional-group churn threshold values and document them here.
- [ ] Decide whether to add an explicit `is_partial_window` flag for rescued
      first-interval churners.
- [ ] State the "conditioned on ≥1 complete interval" scope explicitly in methods.
- [ ] Implement active-period restriction in the interval generator so pre-churn
      quiet intervals do not enter the training frame (separation fix).
- [ ] Verify sensitivity of the whole-dataset imputation: repeat the imputation
      restricted to a training subset and confirm the imputed values (and
      downstream model coefficients) are similar - a brief robustness check
      to document alongside the acknowledged leak.
